# AGFN Hall-of-Fame Visualizer — Tutorial

Turn a finished (or in-progress) **de novo docking** run into figures. This notebook reads the
artifacts written during training and answers three questions:

1. **How did training go?** — plot key curves from `metrics.csv`.
2. **What did it make?** — draw the top-`N` highest-**reward** molecules in 2D.
3. **How do they bind?** — show each molecule's 3D docked pose inside the target pocket
   (a port of Section 4c of `RxnFlow/notebooks/05_visualize_docking_poses.ipynb`).

**Files this notebook reads** (written by training):

| File | Where | What |
|---|---|---|
| `top100_by_reward.sdf` / `top100_by_affinity.sdf` | parent log dir | top-100 molecules + **3D docked poses** + SD tags (`reward`, `docking_score`, `iteration`, `rank`) |
| `top100_by_reward.csv` / `top100_by_affinity.csv` | parent log dir | the same hall of fame as flat tables |
| `metrics.csv`, `config.json` | run subfolder | per-step metrics; run config (target + docking box) |

The hall-of-fame `top100_*` files live in the **stable parent** log dir (cumulative across runs);
`metrics.csv`/`config.json` live in the **timestamped run subfolder**. The setup cell handles both.

> **Prerequisite:** `py3Dmol` for the 3D viewer. If missing: `pip install py3Dmol` into the
> `agfn` env, then restart the kernel.

## Section 0 — Setup

Edit `LOG_DIR` to point at your run's **parent** log directory (the one holding the `top100_*` files, e.g. `AGFN_logs/debug_run`). `N` is how many top-reward molecules to draw/dock-view; `POCKET_DIST` is the Å cutoff for the pocket surface.

In [ ]:
import os, sys
from pathlib import Path

# The only boilerplate: make `import nbtools` resolve, then bootstrap the repo (chdir + src on
# path). Importing nbtools loads py3Dmol FIRST, so rdkit.Chem.Draw (used below) stays safe -- this
# replaces the old "import py3Dmol before Draw" dance.
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "nbtools").is_dir():
        sys.path.insert(0, str(_c)); break
    if (_c / "notebooks" / "nbtools").is_dir():
        sys.path.insert(0, str(_c / "notebooks")); break
import nbtools
from nbtools import hall_of_fame, render2d, render3d, sampling, gif
_root = nbtools.setup_repo()
REPO_ROOT = Path(os.environ.get("AGFN_REPO_ROOT", str(_root)))
print("repo root:", REPO_ROOT)

In [ ]:
# --- knobs (override LOG_DIR without editing via AGFN_VIS_LOG_DIR) -----------------------------
LOG_DIR = Path(os.environ.get(
    "AGFN_VIS_LOG_DIR",
    "/home/aid/dkoh/log/generative/BBB/lrrk2_RTB_Denovo_20260601_173926/"))
N = 10                 # how many top-reward molecules to visualize
POCKET_DIST = 5.0      # Angstrom cutoff for pocket residues / surface
RANK_BY = "reward"     # "reward" or "affinity" -> which top100_by_*.{sdf,csv} to read
# ----------------------------------------------------------------------------------------------
assert LOG_DIR.exists(), f"LOG_DIR not found: {LOG_DIR}"

# Resolve every file we need: auto-find the run subfolder with metrics.csv, the top100_* hall of
# fame (run subdir first, parent as fallback), and the receptor + docking box from config.json.
paths = hall_of_fame.resolve_run_paths(LOG_DIR, REPO_ROOT, rank_by=RANK_BY)

## Section 1 — Training curves from `metrics.csv`

`metrics.csv` has one row per training step. We plot a curated subset versus `Train_iter`,
auto-skipping any column that isn't present so this works across configs. Watch
**`Percent_novel_cumulative`**: it starts near 100% and *falls* as the model rediscovers the
same molecules — the diversity signal that the per-batch `Percent_unique_in_batch` (now a true
percentage over the full sampled batch) can't show once the generator is confident.

Note: some headers carry trailing spaces (e.g. `"Total loss "`); we match by stripped name.

In [ ]:
# Curated training curves vs Train_iter; any column not present in this run is skipped automatically.
if paths.metrics_csv.exists():
    metrics_df = hall_of_fame.plot_metrics(paths.metrics_csv, smooth=5)
else:
    print("metrics.csv not found — run training first.")

### Hall-of-fame curves — top-10 / top-100 mean reward & docking score

These track the **running** top-10 / top-100 generated molecules over training, so they improve
(near-)monotonically: reward goes **up**, docking score goes **down** (more negative kcal/mol =
better binding). The hall of fame only ever swaps a molecule out for a better one, so its averages
can't regress.

- If the run logged the true `Top-10/Top-100 mean reward|docking score` columns (added to
  `metrics.csv` during training), we plot those directly vs `Train_iter`.
- Otherwise we **reconstruct** an approximate curve from the final `top100_by_*.csv` using each
  molecule's discovery `iteration`. This uses only the final survivors (hindsight), so the top-10
  line is a clean lower bound while the top-100 line is a running mean until 100 molecules exist.

In [ ]:
# Running top-10 / top-100 mean reward & docking score: the true logged curves if the run wrote
# them, otherwise reconstructed from the final top100_*.csv files (uses survivors only).
hall_of_fame.plot_topk_curves(paths)

## Section 2 — Top-`N` highest-reward molecules (2D)

We read `top100_by_reward.sdf` (poses + SD tags), take the first `N`, and draw clean 2D
depictions (rebuilt from each record's `smiles` tag), captioned with reward and docking score.

In [ ]:
# Top-N highest-reward molecules as clean 2D depictions (rebuilt from each record's smiles tag),
# captioned with rank / reward / docking score, plus the flat table.
from IPython.display import display
hof = hall_of_fame.load_hall_of_fame(paths.sdf_path, n=N)
print(f"loaded {len(hof)} molecules from {paths.sdf_path.name}")
img = hall_of_fame.hall_of_fame_2d_grid(hof)
if img is not None:
    display(img)
    display(hall_of_fame.hall_of_fame_table(paths, hof, n=N))
else:
    print("No SDF yet — run training to produce the hall of fame.")

## Section 3 — 3D docked pose in the target pocket

A `py3Dmol` grid, **one molecule per cell**: the protein as a gray cartoon, the ligand as
colored sticks, and a translucent **pocket surface** over residues within `POCKET_DIST` Å of
that molecule's pose.

**Why this just works (coordinate frames).** The SDF stores the *ligand only*, but Uni-Dock
writes its atoms at **absolute coordinates in the receptor's frame**. So loading the receptor and
the ligand into the same scene places the ligand in the pocket with no alignment step.

**Two AGFN adaptations** vs RxnFlow's PDB-based notebook:
1. AGFN's `.pdbqt` receptor has a **blank chain column**, so pocket selections use `resi` only.
2. A tiny `pdbqt_to_pdb_text()` converts the receptor to minimal PDB so 3Dmol renders a clean
   cartoon. Pocket residues are resolved **in Python** (not via a fragile 3Dmol `within`
   selector), exactly as RxnFlow does.

In [ ]:
# 3D docked pose grid: gray protein cartoon, cyan-carbon ligand sticks, translucent pocket surface.
# The SDF stores ligand atoms in the receptor frame, so no alignment step is needed.
pocket = render3d.load_receptor_pocket(paths.receptor_path)
if hof and pocket.available:
    render3d.render_pose_grid(hof[:N], pocket, pocket_dist=POCKET_DIST).show()
elif not hof:
    print("No SDF yet — run training to produce the hall of fame.")
else:
    print(f"Receptor not found at {paths.receptor_path}; skipping 3D poses.")

## Section 4 — A-GFN molecular composition steps

A-GFN builds each molecule **incrementally**: a trajectory of graph-edit actions (add atom, add
bond, set charge, …) ending in a *Stop*. Here we re-load the trained model, sample a fresh batch,
**rank it by reward**, and render the step-by-step construction of the best molecule — a faithful
GFlowNet trajectory of a high-reward molecule.

> ⚠️ **Live model.** Unlike the sections above (which only read saved artifacts), this loads the
> trained checkpoint and runs **Uni-Dock docking** on the sampled batch to score/rank it — so it
> needs a **GPU** and the docking backend. Trajectories are stochastic and aren't persisted, so
> this won't reproduce a *specific* saved hall-of-fame SMILES; it shows a top molecule from a
> fresh sample. Set `RANK_INDEX` (0 = best) to walk down the fresh top-K, or
> `AGFN_VIS_CHECKPOINT` / `AGFN_VIS_N_SAMPLES` to override the checkpoint / batch size.

In [ ]:
# Live model: load the FINE-TUNED checkpoint (newest model_state_*.pt in the run subdir; override
# via AGFN_VIS_CHECKPOINT), falling back to the pretrained base when the run has none. hps come from
# denovo.yml (same literals as training). Unlike the sections above, this loads the model and DOCKS.
N_SAMPLES  = int(os.environ.get("AGFN_VIS_N_SAMPLES", "32"))   # trajectories to draw, then rank
RANK_INDEX = 0                                                 # 0 = best; pick within the fresh top-K
DEVICE_ID  = 0
_ckpts = sorted(paths.run_subdir.glob("model_state_*.pt"), key=lambda p: p.stat().st_mtime)
CHECKPOINT = os.environ.get("AGFN_VIS_CHECKPOINT") or (str(_ckpts[-1]) if _ckpts else None)

hps, conditional_range_dict, cond_prop_var = nbtools.load_denovo_hps(REPO_ROOT / "src/config/denovo.yml")
if CHECKPOINT is None:
    CHECKPOINT = hps.saved_model_path          # pretrained base when the run has no checkpoint
hps.saved_model_path = CHECKPOINT
gfn_samples_path = f"{hps.gfn_samples_path}/GFN_gen_samples_{hps.target_name}/"
print("checkpoint:", CHECKPOINT)
finetuner = sampling.build_finetuner(
    hps, conditional_range_dict, cond_prop_var, CHECKPOINT,
    rank=DEVICE_ID, world_size=1, gfn_samples_path=gfn_samples_path)
print("loaded model on", finetuner.device)

In [ ]:
# Sample a fresh batch, rank by reward (replicates samp_iter_finetune.py -> this DOCKS the batch),
# and render the selected molecule's build steps. traj[k][0] is the pre-action graph at step k.
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw
trajs = sampling.sample_trajectories(finetuner, N_SAMPLES)
ranked = sampling.rank_batch_by_reward(finetuner, trajs, task=hps.task)
valid, mols, flat, affinity, order = (ranked["valid"], ranked["mols"], ranked["flat"],
                                      ranked["affinity"], ranked["order"])
print(f"sampled {len(trajs)} trajectories, {len(valid)} valid+renderable")
sel = int(order[min(RANK_INDEX, len(order) - 1)])
print(f"selected fresh rank #{RANK_INDEX}:  reward={flat[sel]:+.3f}  docking={affinity[sel]:+.2f} kcal/mol")
print(f"  SMILES: {Chem.MolToSmiles(mols[sel])}")
img, nsteps = render2d.trajectory_step_grid(valid[sel]["traj"], finetuner.ctx)
print(f"{nsteps} renderable composition steps (t=0 -> final)")
if img is not None:
    display(img)
print("Final molecule:")
display(Draw.MolToImage(mols[sel], size=(360, 300)))

### Section 4b — Animated build GIF

The grid above is a handy contact sheet, but the construction reads more naturally as an
**animation**. `trajectory_to_gif` turns a trajectory (the `(graph, action)` steps in
`valid[sel]["traj"]`) into a looping GIF where the molecule **grows in place**: every atom is
pinned to the position it occupies in the finished molecule, so nothing jumps or rescales as the
structure accretes. Each frame is captioned with the build action (`AddNode C`, `AddEdge (ring)`,
`Set bond type=DOUBLE`, …) and the atom/bond changed at that step is highlighted in orange.

Two details make the in-place growth work (see the code comments):

- **Stable atom identity.** `ctx.graph_to_mol` canonicalises each step through a SMILES
  round-trip that *reorders* atoms, so an atom's index is not comparable across frames — and a
  substructure match would pick an arbitrary embedding, making symmetric rings (e.g. a benzene)
  flip between frames. Instead we stamp every atom with its graph **node label** as an atom-map
  number; map numbers survive the round-trip, so each drawn atom is traced back to its node and
  inherits that node's final 2D coordinate. No matching, no ambiguity.
- **Fixed viewport.** `MolDraw2DCairo.SetScale(...)` locks the world→pixel mapping to the final
  molecule's bounding box, so RDKit doesn't re-centre/zoom each partial structure.

The GIF is written next to the run (`RUN_SUBDIR`) and displayed inline. Re-run the sampling cell
with a different `RANK_INDEX` to animate other molecules from the fresh top-K, or call
`trajectory_to_gif(t["traj"], finetuner.ctx, "out.gif")` on any sampled trajectory `t`.

In [ ]:
# Animate the selected trajectory as a "growing molecule" GIF (written next to the run). Re-run the
# sampling cell with a different RANK_INDEX to animate other molecules from the fresh top-K, or call
# gif.trajectory_to_gif(t["traj"], finetuner.ctx, "out.gif") on any sampled trajectory t.
from IPython.display import Image as IPyImage
if valid:
    gif_path = paths.run_subdir / f"build_traj_rank{RANK_INDEX}.gif"
    gif.trajectory_to_gif(valid[sel]["traj"], finetuner.ctx, gif_path)
    print(f"wrote {gif_path}")
    display(IPyImage(filename=str(gif_path)))
else:
    print("Run the sampling cell above first (need valid / sel).")

In [ ]:
# Optional publication-style 3-panel summary (reward / docking+composite / BBB+solubility). Columns
# absent in this run are skipped; set FONT_DIRS to register custom fonts and OUT_SVG to save.
FONT_DIRS = None   # e.g. ["/path/to/PlusJakartaSans", "/path/to/JetBrainsMono"]
OUT_SVG   = None   # e.g. "./agfn_lrrk2.svg"
if paths.metrics_csv.exists():
    import pandas as pd
    hall_of_fame.plot_training_summary(
        pd.read_csv(paths.metrics_csv), out_svg=OUT_SVG, font_dirs=FONT_DIRS,
        title="A-GFN Training: Reward = QED x SaS x Docking")